In [22]:
import subprocess
import os
import csv
import pandas as pd
# from tqdm import tqdm
# from datetime import datetime
from bayes_opt import BayesianOptimization
from pathlib import Path

In [23]:
#VOT config
#alphaWalk,alphaBike,alphaCarDriver,alphaCarPassenger,alphaBus,alphaTrain,betaChangesTransport,betaCostLow,betaCostMed,betaCostHigh,votCommuteWalk,votCommuteBike,votCommuteCar,votCommuteBus,votCommuteTrain,votOtherWalk,votOtherBike,votOtherCar,votOtherBus,votOtherTrain,carCostKm,ptCostKm,ptBaseCost,weightAccessEgress
defaultAlpha_bounds  = (-5,5)
default_beta_bounds = (-1, -0.001)
pbounds = {'alphaWalk' :defaultAlpha_bounds ,
           'alphaBike' : defaultAlpha_bounds,
           'alphaCarDriver' : defaultAlpha_bounds,
           'alphaCarPassenger' : defaultAlpha_bounds,
           'alphaBus' : defaultAlpha_bounds,
           'alphaTrain' : defaultAlpha_bounds,
           'betaChangesTransport': default_beta_bounds,
           'betaCostLow':(-0.316,-0.316) ,
           'betaCostMed': (-0.2,-0.2),
           'betaCostHigh': (-0.190,-0.190),
           'weightWalk': (1,1),
           'weightWait': (1,1),
           'weightFeeder': (1,1), 
           'weightVotCosts' : (1,1),
           'weightTangibleCosts' : (1,1)
           } 

In [24]:
def writeVotConfigFile(
        alphaWalk,
        alphaBike,
        alphaCarDriver,
        alphaCarPassenger,
        alphaBus,
        alphaTrain,
        betaChangesTransport,
        betaCostLow,
        betaCostMed,
        betaCostHigh,
        weightWalk,
        weightWait,
        weightFeeder,
        weightVotCosts,
        weightTangibleCosts,
        filename) -> int:
    data = {'alphaWalk' : alphaWalk,
                'alphaBike' :alphaBike,
                'alphaCarDriver': alphaCarDriver,
                'alphaCarPassenger':alphaCarPassenger,
                'alphaBus':alphaBus,
                'alphaTrain' :alphaTrain,
                'betaChangesTransport':betaChangesTransport,
                'betaCostLow' :betaCostLow, # betaCostLow,
                'betaCostMed' : betaCostMed,
                'betaCostHigh': betaCostHigh, #betaCostHigh,
                'votCommuteWalk': 15.89,
                'votCommuteBike': 10.17,
                'votCommuteCar': 10.78,
                'votCommuteBus': 7.62,
                'votCommuteTrain': 12.05,
                'votOtherWalk': 11.76,
                'votOtherBike': 10.43,
                'votOtherCar':9.60,
                'votOtherBus':6.66,
                'votOtherTrain':8.64,
                'carCostKm':0.25,
                'ptCostKm':0.187,
                'ptBaseCost':1.08,
                'weightWalk': weightWalk,# 2.0,
                'weightWait': weightWait, # 2.5,
                'weightFeeder': weightFeeder, # 2.0,
                'weightVotCosts' : weightVotCosts,
                'weightTangibleCosts' : weightTangibleCosts
                }
    if Path(filename).exists():
        config = pd.read_csv(filename)
        new_entry = pd.DataFrame([data])


        #config = pd.concat([config, new_entry], ignore_index=True)

        row_count = len(config) #Length off by one, so new index

        new_entry.to_csv(filename,mode='a',header=(not os.path.exists(filename)) , index = None)

        return row_count
    else:
        cols = [
        'alphaWalk', 'alphaBike', 'alphaCarDriver', 'alphaCarPassenger', 'alphaBus', 
        'alphaTrain', 'betaChangesTransport', 'betaCostLow', 'betaCostMed', 'betaCostHigh',
        'votCommuteWalk', 'votCommuteBike', 'votCommuteCar', 'votCommuteBus', 'votCommuteTrain',
        'votOtherWalk', 'votOtherBike', 'votOtherCar', 'votOtherBus', 'votOtherTrain',
        'carCostKm', 'ptCostKm', 'ptBaseCost', 'weightWalk','weightWait','weightFeeder', 'weightVotCosts',
        'weightTangibleCosts'
            ]
        
        df = pd.DataFrame([data], columns=cols)


        df.to_csv(filename, index=None)
        return 0
    
    

### Prepare the Java command

In [25]:
def callSimulation(
        alphaWalk,
        alphaBike,
        alphaCarDriver,
        alphaCarPassenger,
        alphaBus,
        alphaTrain,
        betaChangesTransport,
        betaCostLow,
        betaCostMed,
        betaCostHigh,
        weightWalk,
        weightWait,
        weightFeeder,
        weightVotCosts,
        weightTangibleCosts
        ):
    
    filename = "../7-simulation-Sim-2APL/src/main/resources/baseline_parameterset/vot_parameterset_bayesian.csv"
    parameter_set_index = writeVotConfigFile(
        alphaWalk,
        alphaBike,
        alphaCarDriver,
        alphaCarPassenger,
        alphaBus,
        alphaTrain,
        betaChangesTransport,
        betaCostLow,
        betaCostMed,
        betaCostHigh,
        weightWalk,
        weightWait,
        weightFeeder,
        weightVotCosts,
        weightTangibleCosts,
        filename
        )
    os.chdir("C:\\Users\\ed\\Development\\dhzw\\8-sensitivity-analysis")

    df = pd.DataFrame(columns=["CAR_DRIVER", "CAR_PASSENGER", "BIKE", "BUS_TRAM", "TRAIN", "WALK"])

    df.to_csv("output/output_proportions_" + str(parameter_set_index) + ".csv", index=False)

    config = "--config src/main/resources/config_DHZW_full.toml"
    parameters_path = "--parameter_file src/main/resources/baseline_parameterset/vot_parameterset_bayesian.csv"
    output_path = f'--output_file src/main/resources/baseline_parameterset/output_proportionsCOPY.csv'#../../8-sensitivity-analysis/output/iterations/output_proportions_{parameter_set_index}.csv'
    arg = f'--parameterset_index {parameter_set_index}'
    java_folder_path = '7-simulation-Sim-2APL'
    #print(os.getcwd())
    # set current directory the folder of Sim2APL so I can execute the jar with the correct paths
    if (os.path.basename(os.getcwd()) != java_folder_path):
        new_directory = os.path.join(os.getcwd(), "../", java_folder_path)
        new_directory = new_directory.replace('\\', '/')
        os.chdir(new_directory)
    #print(os.getcwd())

    # parameterset to use
    full_command = f'java -cp target/sim2apl-dhzw-simulation-1.0-SNAPSHOT-jar-with-dependencies.jar main.java.nl.uu.iss.ga.Simulation {config} {output_path} {parameters_path} {arg}'
    #print(full_command)
    try:
        output = subprocess.check_output(
            full_command, stderr=subprocess.STDOUT, universal_newlines=True)
        #print(output)
        return -float(output)
    except subprocess.CalledProcessError as e:
        print(f"Java program exited with non-zero return code: {e.returncode}")
        print(f"Error message: {e.output}")
        exit(1)
    return -999999999999 #an egregiously bad option.

In [26]:
import os
import subprocess


jdk_11_path = r"C:\Program Files\Java\jdk-11"

os.environ["JAVA_HOME"] = jdk_11_path
bin_path = os.path.join(jdk_11_path, "bin")
os.environ["PATH"] = bin_path + os.pathsep + os.environ.get("PATH", "")

# 4. Verification
try:
    result = subprocess.run(["java", "-version"], capture_output=True, text=True, check=True)
    print("Java Version Output:\n", result.stderr) 
except subprocess.CalledProcessError as e:
    print(f"Error: {e}")
except FileNotFoundError:
    print("Java executable not found in the updated PATH.")

Java Version Output:
 openjdk version "11.0.20.1" 2023-08-24 LTS
OpenJDK Runtime Environment Microsoft-8297088 (build 11.0.20.1+1-LTS)
OpenJDK 64-Bit Server VM Microsoft-8297088 (build 11.0.20.1+1-LTS, mixed mode)



In [27]:
optimizer = BayesianOptimization(
    f=callSimulation,
    pbounds=pbounds,
    random_state=1,
)

In [28]:
# optimizer.maximize(
#     init_points=140,
#     n_iter=100,
# )

optimizer.maximize(
    init_points=140,
    n_iter=100,
)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaCh... | betaCo... | betaCo... | betaCo... | weight... | weight... | weight... | weight... | weight... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -33.04785 | -0.829779 | 2.2032449 | -4.998856 | -1.976674 | -3.532441 | -4.076614 | -0.813926 | -0.316    | -0.2      | -0.19     | 1.0       | 1.0       | 1.0       | 1.0       | 1.0       |
| 2         | -21.75416 | 1.7046751 | -0.826951 | 0.5868982 | -3.596130 | -3.018985 | 3.0074456 | -0.032706 | -0.316    | -0.2      | -0.19     | 1.0       | 1.0       | 1.0       | 1.0       | 1.0       |
| 3         | -13.27361 | -4.016531 | -0.788923 | 4.5788953 | 0.3316528 | 1.9187711 | -1.844843 | -0.314185 | -0.316    | -0.2      | -0.19     | 1.0       | 1.0       | 1.0   

In [29]:
print(optimizer.max)

{'target': np.float64(-3.744639821651025), 'params': {'alphaWalk': np.float64(-5.0), 'alphaBike': np.float64(-5.0), 'alphaCarDriver': np.float64(1.3649139031623834), 'alphaCarPassenger': np.float64(-2.217672850066221), 'alphaBus': np.float64(-3.1153302363175612), 'alphaTrain': np.float64(-4.205558892086803), 'betaChangesTransport': np.float64(-0.001), 'betaCostLow': np.float64(-0.316), 'betaCostMed': np.float64(-0.2), 'betaCostHigh': np.float64(-0.19), 'weightWalk': np.float64(1.0), 'weightWait': np.float64(1.0), 'weightFeeder': np.float64(1.0), 'weightVotCosts': np.float64(1.0), 'weightTangibleCosts': np.float64(1.0)}}


In [30]:
callSimulation(**optimizer.max['params'])

-3.7585991594418524

### Main loop to call the simulations

#20 iterations of sequential. 
{'target': np.float64(-1618722.5288986785), 'params': {'alphaWalk': np.float64(-10.0), 'alphaBike': np.float64(-3.196817588926173), 'alphaCarDriver': np.float64(4.888472151768744), 'alphaCarPassenger': np.float64(10.0), 'alphaBus': np.float64(6.829128765087262), 'alphaTrain': np.float64(10.0), 'betaChangesTransport': np.float64(0.001), 'betaCostLow': np.float64(1.0), 'betaCostMed': np.float64(1.0), 'betaCostHigh': np.float64(1.0), 'weightWalk': np.float64(1.0), 'weightWait': np.float64(3.0), 'weightFeeder': np.float64(1.0)}}

#20 iterations of aggregate.
{'target': np.float64(-1805588.612258473), 'params': {'alphaWalk': np.float64(-10.0), 'alphaBike': np.float64(-5.075847260627779), 'alphaCarDriver': np.float64(3.316409577447148), 'alphaCarPassenger': np.float64(10.0), 'alphaBus': np.float64(-0.3293318074942246), 'alphaTrain': np.float64(10.0), 'betaChangesTransport': np.float64(0.001), 'betaCostLow': np.float64(0.001), 'betaCostMed': np.float64(1.0), 'betaCostHigh': np.float64(1.0), 'weightWalk': np.float64(0.0), 'weightWait': np.float64(2.2692369147311315), 'weightFeeder': np.float64(3.0)}}